# Does NIFTY recover after a significant one-day fall?

**Hypothesis (informal):** *After a significant one-day fall in NIFTY, the market tends to recover over the next few trading days.*

This notebook runs top-to-bottom. Every research rule is fixed in Section 3 **before** any outcome is computed, and all decisions (with reasons and alternatives) are logged in `docs/DECISIONS.md`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import load_clean

pd.set_option("display.width", 140)
RAW_PATH = ROOT / "data" / "raw" / "nifty50_niftyindices_raw.csv"

## 1. Data source

| Item | Value |
|---|---|
| Source | NIFTY Indices (NSE Indices Ltd) — official index provider — https://www.niftyindices.com/reports/historical-data |
| Series | NIFTY 50 price index, daily |
| Fields | Date, Open, High, Low, Close |
| Requested range | 01-Jan-2000 → latest available |
| How obtained | `python src/fetch_data.py` — calls the site's own backend endpoint; the UI's 1-year-per-query limit is a client-side check only |
| Raw file | `data/raw/nifty50_niftyindices_raw.csv`, saved exactly as returned; never edited |

**Why 2000 onwards:** more observations and several different market regimes (2000–01 dot-com crash, 2008 crisis, 2020 COVID crash). The period was chosen for coverage, before looking at any results.

## 2. Data validation

Checks: unparseable dates, missing or non-positive prices, duplicate dates, ordering, OHLC consistency (High ≥ Open/Close ≥ Low), calendar gaps, trading days per year, suspicious Open prices, and largest moves.

**Cleaning policy**
- **Invalid rows → the code raises an error** instead of silently dropping them (none were found).
- **Ordering:** the source delivers newest-first; data is re-sorted ascending.
- **Calendar gaps are reported, not filled.** Holidays are not missing data. All gaps > 4 days are 5–6 days (a holiday next to a weekend), and every full year has 243–254 trading days, which rules out isolated missing days.
- **Extreme days are kept.** The largest moves are real events (17-May-2004 election shock, 24-Oct-2008, 23-Mar-2020 COVID, 18-May-2009 election rally), and they are exactly what the hypothesis is about. A sensitivity check later excludes the 5 events with the largest absolute forward return.
- **Suspicious Opens are flagged, not changed** (see below).

In [2]:
df, report = load_clean(RAW_PATH)
print(report.summary())
report.gaps

Rows: 6644  |  Coverage: 2000-01-03 -> 2026-09-21
Source sorted ascending: False (re-sorted ascending)
Calendar gaps > 4 days: 12 (max 6 days)
Trading days per full year: min 243, max 254
Validation errors: 0


,from,to,days
0,2000-03-16,2000-03-21,5.0
1,2005-11-02,2005-11-07,5.0
2,2008-03-19,2008-03-24,5.0
3,2009-04-29,2009-05-04,5.0
4,2009-12-24,2009-12-29,5.0
5,2012-04-04,2012-04-09,5.0
6,2014-10-01,2014-10-07,6.0
7,2015-04-01,2015-04-06,5.0
8,2016-03-23,2016-03-28,5.0
9,2016-04-13,2016-04-18,5.0


In [3]:
# Largest one-day moves: are they data errors or real events?
report.largest_moves

,Date,ret
0,2020-03-23,-0.129805
1,2004-05-17,-0.122377
2,2008-10-24,-0.122029
3,2008-01-21,-0.087024
4,2020-03-12,-0.083019
5,2009-05-18,0.177441
6,2020-04-07,0.087632
7,2004-05-18,0.082952
8,2000-04-07,0.071716
9,2008-10-31,0.069910


### Suspicious Open prices (pre-2011)

A genuine opening price almost never equals exactly the previous day's Close, and it rarely sits exactly at the day's High or Low. Both patterns are common in 2000–2010 and nearly absent afterwards, which suggests that older Opens are often stale or placeholder values.

This matters because the realistic entry is the **next day's Open**. Decision: keep the full dataset, use next-day Open as the main entry, and repeat the analysis with a **next-day Close** entry as a robustness check. Each row carries a `suspicious_open` flag.

In [4]:
report.suspicious_open_by_year.style.format({"open_eq_prev_close": "{:.1%}", "open_at_high_or_low": "{:.1%}"})

,days,open_eq_prev_close,open_at_high_or_low
Date,,,
2000,250,5.6%,26.0%
2001,248,7.7%,21.4%
2002,251,10.0%,21.1%
2003,254,11.4%,18.1%
2004,254,5.1%,12.2%
2005,251,6.8%,11.2%
2006,250,5.2%,19.2%
2007,249,4.8%,15.7%
2008,246,2.0%,23.6%


## 3. Research design (fixed before any outcomes are computed)

### 3.1 Significant fall (event definition)

Day *t* is an **event** if its close-to-close return is below **−2 × σ**, where σ is the standard deviation of daily close-to-close returns over the **previous 60 trading days** (days *t−60 … t−1*). Day *t* itself is excluded from σ, so there is no look-ahead.

**Why volatility-scaled instead of a fixed −2%:** a −2% day in 2008 (daily swings of ±3%) is ordinary, while a −2% day in 2017 (swings of ±0.5%) is extreme. Counting events only, without outcomes, showed that a fixed −2% rule puts 61 events in 2008 alone and none in 2017 or 2023, so it mainly identifies turbulent years. Scaling makes "significant" mean *unusual relative to current conditions*.

Note: σ measures the *size* of recent moves, not their direction. A slow, steady decline has low σ, so a sudden large fall during it still counts.

*Example:* σ over the last 60 days = 1.0% → event if today's return < −2.0%. If σ = 3.0% → event only if return < −6.0%.

**Robustness:** 1.5× and 2.5× σ, fixed −2%, 20-day σ window.

### 3.2 Recovery (descriptive measure, main)

**50% retracement, measured on daily Closes.**

```
loss           = previous_day_close − event_day_close
recovery_level = event_day_close + 0.50 × loss
recovered      = any daily Close within the holding period ≥ recovery_level
```

*Worked example*

| | Price |
|---|---|
| Previous-day Close | ₹20,000 |
| Event-day Close | ₹19,000 |
| Loss | ₹1,000 |
| 50% of loss | ₹500 |
| **Recovery level** | **₹19,500** |

| Day after event | Close | Recovered? |
|---|---|---|
| Day 1 | ₹19,300 | No |
| Day 2 | ₹19,500 | **Yes**, level reached |
| Day 3 | ₹19,700 | Yes (already recovered on day 2) |

**Why daily Close, not High:** a brief intraday touch is not a recovery, and using Close keeps the measure consistent with the close-based event definition.

**Baseline:** the same recovery measurement applied to **non-event days** (days that do not meet the 2× σ rule, whether they were small down, flat or up days). The evidence is the *comparison*, e.g. event recovery 60% vs baseline 45%. A high event recovery rate on its own means nothing if ordinary days recover just as often.

**Robustness:** 100% (full) retracement, meaning a later Close ≥ the previous-day Close (₹20,000 in the example).

**Recovery ≠ tradable return.** A recovery can happen overnight. If the event-day Close is ₹19,000 and the next Open is already ₹20,000, the market *has* recovered, but a trader buying at that Open captured none of it. So the recovery test answers *"does the market bounce?"*, and **forward returns from a realistic entry price are computed separately** to answer *"could you profit from it?"*

#### 3.2a Baseline recovery target: matched in volatility units

A normal day has no "loss", so it needs an equivalent target. Each event's target is converted into multiples of recent volatility σ (the same 60-day σ as the event rule) and applied to normal days using **their own** σ:

```
k_i          = (recovery_level_i / event_close_i − 1) / σ_i        # event i's target, in σ units
target_j     = close_j × (1 + k_i × σ_j)                           # same target for normal day j
baseline_i   = share of normal days (same sample period) whose Close reaches target_j within N days
baseline     = average of baseline_i across all events
```

*Worked example:* the event needs +2.6% to reach its 50% level, and its σ = 1.8%, so k = 2.6 / 1.8 ≈ **1.44σ**.
- A normal day in a calm period (σ = 0.6%) must rise 1.44 × 0.6% ≈ **+0.9%**.
- A normal day in a turbulent period (σ = 3.0%) must rise 1.44 × 3.0% ≈ **+4.3%**.

**Why:** a +2.6% rise is easy when daily swings are ±3% and hard when they are ±0.5%. Since events are defined in σ units (2σ), the comparison should be in σ units too, so that event and baseline face equally difficult targets in every volatility regime.

### 3.3 Holding period

**Main: N = 5 trading days** (about one trading week, the closest match to "the next few trading days").

Counting convention: the event is day *t*; "within N days" means the Closes of **t+1, t+2, ..., t+N**. For N = 5 the window is exactly the next 5 trading sessions. The same N is used for the recovery window and for the forward-return horizon.

*Example:* event on Monday → window = Tuesday, Wednesday, Thursday, Friday and the following Monday (holidays are skipped automatically, because we count trading days, not calendar days).

**The clock starts at the event, always.** We do **not** wait for the first up day and then start counting. That rule would depend on the price path after the event, so the window would start only once the bounce is already under way, and recovery rates would be biased upward.

**Robustness:** N = 1, 3, 10. N = 5 is fixed in advance as the main horizon so that the horizon cannot be tuned to the best-looking result.

### 3.4 – 3.9 Remaining design decisions

*Pending: exit rule, overlapping events, baseline construction for forward returns, development / out-of-sample split, transaction costs, statistical method, and the rejection criterion.*